# Time-Series EDA

Phase 2 notebook for trend checks, missing values, and injected anomaly visibility.

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path('../data/raw')
if not DATA_DIR.exists():
    DATA_DIR = Path('data/raw')

paths = sorted(DATA_DIR.glob('timeseries_*.csv'))
frames = [pd.read_csv(path, parse_dates=['date']) for path in paths]
timeseries = pd.concat(frames, ignore_index=True)
timeseries.head()

In [ ]:
print('Shape:', timeseries.shape)
display(timeseries.info())
display(timeseries.isna().sum())
display(timeseries.groupby('company_name')['date'].agg(['min', 'max', 'count']))

In [ ]:
summary = timeseries.groupby('company_name').agg(
    injected_anomalies=('is_injected_anomaly', 'sum'),
    mean_revenue=('revenue', 'mean'),
    min_revenue=('revenue', 'min'),
    max_revenue=('revenue', 'max'),
    mean_active_users=('active_users', 'mean'),
    mean_signups=('signups', 'mean'),
).round(2)
display(summary)

In [ ]:
for company, company_df in timeseries.groupby('company_name'):
    company_df = company_df.sort_values('date')
    first_revenue = company_df['revenue'].iloc[0]
    last_revenue = company_df['revenue'].iloc[-1]
    revenue_change = (last_revenue / first_revenue - 1) * 100
    print(f'{company}: {revenue_change:.2f}% revenue change')

In [ ]:
for company, company_df in timeseries.groupby('company_name'):
    company_df = company_df.sort_values('date')
    anomalies = company_df[company_df['is_injected_anomaly'] == 1]

    plt.figure(figsize=(12, 4))
    plt.plot(company_df['date'], company_df['revenue'], label='revenue')
    if not anomalies.empty:
        plt.scatter(anomalies['date'], anomalies['revenue'], color='red', label='injected anomaly')
    plt.title(f'{company} Revenue')
    plt.xlabel('date')
    plt.ylabel('revenue')
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
for metric in ['revenue', 'active_users', 'signups']:
    pivot = timeseries.pivot(index='date', columns='company_name', values=metric)
    pivot.plot(figsize=(12, 4), title=f'{metric} by Company')
    plt.xlabel('date')
    plt.ylabel(metric)
    plt.tight_layout()
    plt.show()